In [4]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../..").resolve()))

import pandas as pd
import numpy as np
from loguru import logger

from world_cup_2026.config import RAW_DATA_DIR, RANDOM_SEED
from world_cup_2026.data_ingestion.normalize import normalize_dataframe_teams
from world_cup_2026.features.elo import calculate_elo
from world_cup_2026.features.form import FormCalculator
from world_cup_2026.features.h2h import H2HAnalyzer

np.random.seed(RANDOM_SEED)

df_results = pd.read_csv(
    RAW_DATA_DIR / "martj42_results" / "results.csv",
    parse_dates=["date"]
)
df_results_norm = normalize_dataframe_teams(df_results, ["home_team", "away_team"])

print(f"Matches loaded: {len(df_results_norm):,}")
print(f"Date range: {df_results_norm['date'].min().date()} → {df_results_norm['date'].max().date()}")
print("Setup complete.")

2026-04-03 21:53:20.485 | INFO     | world_cup_2026.config:<module>:12 - PROJ_ROOT path is: C:\Users\feder\Documents\data_repos\world-cup-2026-predictor
2026-04-03 21:53:21.085 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:193 - Unknown team name: 'Guernsey' — add to normalize.py if needed.
2026-04-03 21:53:21.085 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:193 - Unknown team name: 'Jersey' — add to normalize.py if needed.
2026-04-03 21:53:21.103 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:193 - Unknown team name: 'Alderney' — add to normalize.py if needed.
2026-04-03 21:53:21.106 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:193 - Unknown team name: 'Catalonia' — add to normalize.py if needed.
2026-04-03 21:53:21.109 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:193 - Unknown team name: 'Basque Country' — add to normalize.py if needed.
2026-04-03 2

Matches loaded: 49,215
Date range: 1872-11-30 → 2026-03-31
Setup complete.


In [5]:
df_rank = pd.read_csv(RAW_DATA_DIR / "cashncarry_rankings" / "fifa_ranking-2024-06-20.csv")
print(df_rank.shape)
print(df_rank.columns.tolist())
print(df_rank.head(3).to_string())

(67472, 8)
['rank', 'country_full', 'country_abrv', 'total_points', 'previous_points', 'rank_change', 'confederation', 'rank_date']
    rank       country_full country_abrv  total_points  previous_points  rank_change confederation   rank_date
0  140.0  Brunei Darussalam          BRU           2.0              0.0          140           AFC  1992-12-31
1   33.0           Portugal          POR          38.0              0.0           33          UEFA  1992-12-31
2   32.0             Zambia          ZAM          38.0              0.0           32           CAF  1992-12-31


In [8]:
df_rank["rank_date"] = pd.to_datetime(df_rank["rank_date"])
print(f"Date range: {df_rank['rank_date'].min().date()} → {df_rank['rank_date'].max().date()}")
print(f"Unique dates: {df_rank['rank_date'].nunique()}")

# Check name matching against WC2026 teams
df_teams = pd.read_csv(RAW_DATA_DIR / "areezvisram12_fixture" / "teams.csv")
confirmed_teams = df_teams[df_teams["is_placeholder"] == False]["team_name"].tolist()

print(f"\nName matching check ({len(confirmed_teams)} WC2026 teams):")
not_found = []
for team in sorted(confirmed_teams):
    found = df_rank["country_full"].eq(team).any()
    if not found:
        not_found.append(team)
        print(f"  ✗ '{team}'")

if not not_found:
    print("  All teams found directly.")
else:
    print(f"\nTotal not found: {len(not_found)}")

Date range: 1992-12-31 → 2024-06-20
Unique dates: 333

Name matching check (48 WC2026 teams):
  ✗ 'Bosnia-Herzegovina'
  ✗ 'Cape Verde'
  ✗ 'DR Congo'
  ✗ 'Iran'
  ✗ 'South Korea'

Total not found: 5


In [10]:
search_terms = {
    "Bosnia-Herzegovina": "Bosnia",
    "Cape Verde":         "Cape",
    "DR Congo":           "Congo",
    "Iran":               "Iran",
    "South Korea":        "Korea",
}

for our_name, search in search_terms.items():
    matches = df_rank[df_rank["country_full"].str.contains(search, case=False)]["country_full"].unique()
    print(f"'{our_name}' → candidates: {matches}")

'Bosnia-Herzegovina' → candidates: ['Bosnia and Herzegovina']
'Cape Verde' → candidates: []
'DR Congo' → candidates: ['Congo' 'Congo DR']
'Iran' → candidates: ['IR Iran']
'South Korea' → candidates: ['Korea Republic' 'Korea DPR']


In [11]:
# ── Ranking name mapping → our normalized names ───────────────────────────────
RANKING_NAME_MAP = {
    "Bosnia and Herzegovina": "Bosnia-Herzegovina",
    "Congo DR":               "DR Congo",
    "IR Iran":                "Iran",
    "Korea Republic":         "South Korea",
}

df_rank["team"] = df_rank["country_full"].replace(RANKING_NAME_MAP)

# Cape Verde not in rankings dataset — will get NaN, filled with median later
print("Mapping applied.")
print(f"Unique teams in rankings after mapping: {df_rank['team'].nunique()}")

# Verify the 5 problematic teams
for team in ["Bosnia-Herzegovina", "DR Congo", "Iran", "South Korea", "Cape Verde"]:
    found = df_rank["team"].eq(team).any()
    print(f"  '{team}': {'✓' if found else '✗ will be NaN'}")

Mapping applied.
Unique teams in rankings after mapping: 216
  'Bosnia-Herzegovina': ✓
  'DR Congo': ✓
  'Iran': ✓
  'South Korea': ✓
  'Cape Verde': ✗ will be NaN


In [13]:
from world_cup_2026.config import PROCESSED_DATA_DIR

# ── Build ranking lookup ───────────────────────────────────────────────────────
df_rank_clean = (
    df_rank[["rank_date", "team", "rank", "confederation"]]
    .dropna(subset=["rank"])
    .sort_values(["team", "rank_date"])
)

def get_rank_as_of(team, date):
    """Return most recent ranking for team strictly before match date."""
    rows = df_rank_clean[
        (df_rank_clean["team"] == team) &
        (df_rank_clean["rank_date"] < date)
    ]
    if rows.empty:
        return None, None
    last = rows.iloc[-1]
    return last["rank"], last["confederation"]

# ── Apply to master_features ───────────────────────────────────────────────────
df_master = pd.read_parquet(PROCESSED_DATA_DIR / "master_features.parquet")
print(f"Master features loaded: {df_master.shape}")

print("Computing rankings... (this may take ~30s)")

home_ranks, away_ranks = [], []
home_confs, away_confs = [], []

for _, row in df_master.iterrows():
    hr, hc = get_rank_as_of(row["home_team"], row["date"])
    ar, ac = get_rank_as_of(row["away_team"], row["date"])
    home_ranks.append(hr)
    away_ranks.append(ar)
    home_confs.append(hc)
    away_confs.append(ac)

df_master["ranking_home"]      = home_ranks
df_master["ranking_away"]      = away_ranks
df_master["ranking_diff"]      = df_master["ranking_home"] - df_master["ranking_away"]
df_master["confederation_home"] = home_confs
df_master["confederation_away"] = away_confs

print(f"Done.")
print(f"Missing ranking_home: {df_master['ranking_home'].isna().sum()}")
print(f"Missing ranking_away: {df_master['ranking_away'].isna().sum()}")
print(f"\nSample:")
print(df_master[["date","home_team","away_team","ranking_home","ranking_away","ranking_diff"]].head(5).to_string())

Master features loaded: (9796, 89)
Computing rankings... (this may take ~30s)
Done.
Missing ranking_home: 794
Missing ranking_away: 797

Sample:
        date            home_team    away_team  ranking_home  ranking_away  ranking_diff
0 2016-01-03                India  Afghanistan         166.0         150.0          16.0
1 2016-01-06               Rwanda     Cameroon         101.0          59.0          42.0
2 2016-01-06               Sweden      Estonia          35.0          93.0         -58.0
3 2016-01-08           Bangladesh    Sri Lanka         179.0         188.0          -9.0
4 2016-01-08  Trinidad and Tobago        Haiti          50.0          79.0         -29.0


In [14]:
# ── Handle missing rankings ────────────────────────────────────────────────────
# Fill NaN with median ranking overall (conservative — unknown teams are mid-tier)
median_rank = df_master["ranking_home"].median()
print(f"Median ranking: {median_rank:.0f}")

df_master["ranking_home"] = df_master["ranking_home"].fillna(median_rank)
df_master["ranking_away"] = df_master["ranking_away"].fillna(median_rank)
df_master["ranking_diff"] = df_master["ranking_home"] - df_master["ranking_away"]

# Fill missing confederation with "Unknown"
df_master["confederation_home"] = df_master["confederation_home"].fillna("Unknown")
df_master["confederation_away"] = df_master["confederation_away"].fillna("Unknown")

print(f"Missing after fill — ranking_home: {df_master['ranking_home'].isna().sum()}")
print(f"Missing after fill — ranking_away: {df_master['ranking_away'].isna().sum()}")

# ── Save updated master_features ───────────────────────────────────────────────
df_master.to_parquet(PROCESSED_DATA_DIR / "master_features.parquet", index=False)
print(f"\nSaved: {df_master.shape}")
print(f"New columns: {[c for c in df_master.columns if c not in ['date','home_team','away_team','home_score','away_score','tournament','neutral','target','year']][-5:]}")

Median ranking: 76
Missing after fill — ranking_home: 0
Missing after fill — ranking_away: 0

Saved: (9796, 94)
New columns: ['ranking_home', 'ranking_away', 'ranking_diff', 'confederation_home', 'confederation_away']


In [9]:
import time

# --- Elo ---
t0 = time.time()
df_elo = calculate_elo(df_results_norm)
print(f"Elo: {len(df_elo):,} rows, {time.time()-t0:.1f}s")

# --- Form ---
t0 = time.time()
calc = FormCalculator(df_results_norm, windows=[5, 10, 20])
df_form = calc.compute_form_features(df_results_norm)
print(f"Form: {len(df_form):,} rows, {len(df_form.columns)} cols, {time.time()-t0:.1f}s")

# Sanity check
assert len(df_elo) == len(df_form) == len(df_results_norm), "Row count mismatch!"
print("Row count check passed.")

2026-04-03 21:55:13.962 | INFO     | world_cup_2026.features.elo:calculate_elo:196 - Calculating Elo for 49,215 matches...
2026-04-03 21:55:15.217 | SUCCESS  | world_cup_2026.features.elo:calculate_elo:243 - Elo calculation complete. Registry has 333 teams.
2026-04-03 21:55:15.261 | INFO     | world_cup_2026.features.form:__init__:95 - FormCalculator initialized — 49,215 matches, windows=[5, 10, 20], decay_lambda=0.1
2026-04-03 21:55:15.320 | INFO     | world_cup_2026.features.form:compute_form_features:309 - Computing form features (all matches) for 49,215 rows...


Elo: 49,215 rows, 1.4s


2026-04-03 21:55:15.640 | DEBUG    | world_cup_2026.features.form:_build_long_df:155 - Long format built: 98,430 rows, 333 teams.
2026-04-03 21:55:15.643 | DEBUG    | world_cup_2026.features.form:_compute_rolling_features:182 -   Computing rolling window=5...
2026-04-03 21:55:17.336 | DEBUG    | world_cup_2026.features.form:_compute_rolling_features:182 -   Computing rolling window=10...
2026-04-03 21:55:18.724 | DEBUG    | world_cup_2026.features.form:_compute_rolling_features:182 -   Computing rolling window=20...
2026-04-03 21:55:22.592 | SUCCESS  | world_cup_2026.features.form:compute_form_features:358 - Form features computed. Added 66 columns.


Form: 49,215 rows, 75 cols, 7.4s
Row count check passed.


In [ ]:
# --- H2H ---
t0 = time.time()
analyzer = H2HAnalyzer(df_elo)
print(f"H2H analyzer initialized, {time.time()-t0:.1f}s")

# --- Target variable ---
# From home team perspective: 2=home win, 1=draw, 0=away win
def encode_result(row):
    if row["home_score"] > row["away_score"]:
        return 2
    elif row["home_score"] == row["away_score"]:
        return 1
    else:
        return 0

df_results_norm["target"] = df_results_norm.apply(encode_result, axis=1)

print(f"\nTarget distribution:")
print(df_results_norm["target"].value_counts().sort_index().rename({0: "Away win", 1: "Draw", 2: "Home win"}))
print(f"\nHome win rate: {(df_results_norm['target']==2).mean():.3f}")
print(f"Draw rate:     {(df_results_norm['target']==1).mean():.3f}")
print(f"Away win rate: {(df_results_norm['target']==0).mean():.3f}")

In [ ]:
# --- Filter to modern era for feature matrix ---
CUTOFF_DATE = pd.Timestamp("2016-01-01")

df_master = df_elo[ELO_COLS].copy()
df_master["target"] = df_results_norm["target"].values
df_master = pd.concat([df_master, df_form[FORM_COLS].reset_index(drop=True)], axis=1)

# Filter AFTER joining — Elo and Form were computed on full history (correct)
# We only exclude pre-2000 from the training matrix
df_master = df_master[df_master["date"] >= CUTOFF_DATE].reset_index(drop=True)

print(f"Matches from 2000 onwards: {len(df_master):,}")
print(f"Date range: {df_master['date'].min().date()} → {df_master['date'].max().date()}")
print(f"Shape after filter: {df_master.shape}")

# --- H2H features ---
print("\nComputing H2H features...")
t0 = time.time()

h2h_records = []
for idx, row in df_master.iterrows():
    features = analyzer.get_matchup_features(
        team_a=row["home_team"],
        team_b=row["away_team"],
        as_of=row["date"]
    )
    h2h_records.append(features)

    if idx % 5000 == 0 and idx > 0:
        elapsed = time.time() - t0
        pct = idx / len(df_master) * 100
        print(f"  {idx:,}/{len(df_master):,} ({pct:.0f}%) — {elapsed:.1f}s elapsed")

df_h2h = pd.DataFrame(h2h_records)
df_master = pd.concat([df_master, df_h2h.reset_index(drop=True)], axis=1)

elapsed = time.time() - t0
print(f"\nH2H done: {elapsed:.1f}s")
print(f"Final shape: {df_master.shape}")
print(f"H2H columns: {list(df_h2h.columns)}")

In [ ]:
# Drop H2H metadata columns that aren't features
H2H_META_COLS = ["team_a", "team_b", "as_of"]
df_master = df_master.drop(columns=H2H_META_COLS, errors="ignore")

print(f"Shape after cleanup: {df_master.shape}")

# Check NaN counts
nan_counts = df_master.isnull().sum()
nan_cols = nan_counts[nan_counts > 0]
if len(nan_cols) == 0:
    print("No NaN values found.")
else:
    print(f"\nColumns with NaN ({len(nan_cols)} total):")
    print(nan_cols.sort_values(ascending=False).head(20))

# Check target distribution in filtered dataset
print(f"\nTarget distribution (2016+):")
print(df_master["target"].value_counts().sort_index().rename({0: "Away win", 1: "Draw", 2: "Home win"}))
print(f"\nHome win rate: {(df_master['target']==2).mean():.3f}")
print(f"Draw rate:     {(df_master['target']==1).mean():.3f}")
print(f"Away win rate: {(df_master['target']==0).mean():.3f}")

# Preview
print(f"\nColumn groups:")
print(f"  Identity:  {[c for c in df_master.columns if c in ['date','home_team','away_team','tournament','neutral']]}")
print(f"  Elo:       {[c for c in df_master.columns if 'elo' in c or 'win_prob' in c]}")
print(f"  Form home: {len([c for c in df_master.columns if c.startswith('home_form_')])} cols")
print(f"  Form away: {len([c for c in df_master.columns if c.startswith('away_form_')])} cols")
print(f"  H2H:       {[c for c in df_master.columns if c.startswith('h2h_') or c.startswith('transitive_')]}")
print(f"  Target:    {['target']}")

In [ ]:
from world_cup_2026.config import PROCESSED_DATA_DIR

# Create directory if it doesn't exist
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

output_path = PROCESSED_DATA_DIR / "master_features.parquet"
df_master.to_parquet(output_path, index=False)

print(f"Saved: {output_path}")
print(f"File size: {output_path.stat().st_size / 1024:.1f} KB")

# Verify roundtrip
df_check = pd.read_parquet(output_path)
assert df_check.shape == df_master.shape, "Roundtrip shape mismatch!"
assert list(df_check.columns) == list(df_master.columns), "Roundtrip columns mismatch!"
print(f"Roundtrip verification passed.")
print(f"\nFinal feature matrix: {df_master.shape[0]:,} rows x {df_master.shape[1]} columns")
print(f"Ready for modeling.")

In [ ]:
df_rank = pd.read_csv("data/raw/cashncarry_rankings/fifa_ranking-2024-06-20.csv")
print(df_rank.shape)
print(df_rank.columns.tolist())
print(df_rank.head(3).to_string())